In [ ]:
import os
import torch
import scipy.io as sio
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Dataset, Data, InMemoryDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool
from sklearn.model_selection import train_test_split
import torch.optim as optim
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, roc_auc_score, confusion_matrix
)
# NOTE: This is v1 of the pipeline, showing the core longitudinal feature
# engineering approach. Edge construction and graph saving are incomplete here.
# See 02_gat_leakage_corrected_pipeline.ipynb for the complete, corrected pipeline.
class ImprovedBrainNetworkDataset(InMemoryDataset):
    def __init__(self, root, response_path, transform=None, pre_transform=None):
        self.response_path = response_path
        super().__init__(root, transform, pre_transform)
        self.data, self.slices = torch.load(self.processed_paths[0])

    @property
    def processed_file_names(self):
        return ['processed_graphs.pt']
    
    
    def process(self):
        data_list = []
        responses = pd.read_csv(self.response_path)
        responses_dict = dict(zip( responses['patient_id'], responses['label']))

        # Group files by patient
        patient_files = {}
        mat_files = [os.path.join(self.raw_dir, f) for f in os.listdir(self.raw_dir) if f.endswith('.mat')]
        for mat_path in mat_files:
            # File naming convention: MXXXXV.mat (XXXX = patient ID, V = visit number)
            # Adapt this parsing to match your dataset's naming convention
            patient_id = int(os.path.basename(mat_path).split('M')[1][:4])
            visit_num = int(os.path.basename(mat_path).split('M')[1][4])
            if patient_id not in patient_files:
                patient_files[patient_id] = []
            patient_files[patient_id].append((visit_num, mat_path))

        # Process each patient's sequence of visits
        for patient_id, visits in patient_files.items():
            # Sort visits chronologically
            visits.sort()
        
            # Create features showing changes between visits
            visit_features = []
            for i, (visit_num, mat_path) in enumerate(visits):
                mat_data = sio.loadmat(mat_path)
                dti_r = mat_data['structural_connectivity']['r'][0][0]
                rest_r = mat_data['functional_connectivity']['r'][0][0]
                combined_adj = (dti_r + rest_r) / 2
            
                if i > 0:
                    # Calculate change from previous visit
                    change = combined_adj - previous_adj
                    visit_features.append(change)
            
                previous_adj = combined_adj

            if visit_features:  # If we have at least one change measurement
                # Aggregate changes across visits
                avg_change = np.mean(visit_features, axis=0)
            
                # Create graph data with change features
                adj_matrix = torch.tensor(avg_change, dtype=torch.float32)
                # ... rest of graph creation code ...
            
                label = responses_dict.get(patient_id, 0)
                data = Data(
                    x=adj_matrix,
                    edge_index=edge_indices,
                    edge_attr=edge_attr,
                    y=torch.tensor([label], dtype=torch.long)
                )
                data_list.append(data)

    
    
    
    
    
    
    
class ImprovedBrainNetworkGNN(nn.Module):
    def __init__(self, num_features, hidden_channels=64, dropout_rate=0.3):
        super().__init__()
        
        # Graph Attention Layers
        self.gat1 = GATConv(num_features, hidden_channels, heads=4, concat=True)                #2
        self.gat2 = GATConv(hidden_channels*4, hidden_channels//2, heads=2, concat=False)       #2
        
        # Batch Normalization
        self.bn1 = nn.BatchNorm1d(hidden_channels*4)                                            #3
        self.bn2 = nn.BatchNorm1d(hidden_channels//2)                                           #3
        
        # Dropout
        self.dropout = nn.Dropout(dropout_rate)                                                 #3
        
        # Final Classification Layer
        self.fc = nn.Linear(hidden_channels//2, 1)                                              #3

    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        
        # First Graph Attention Layer
        x = self.gat1(x, edge_index, edge_attr)
        x = self.bn1(x)
        x = F.leaky_relu(x)
        x = self.dropout(x)
        
        # Second Graph Attention Layer
        x = self.gat2(x, edge_index, edge_attr)
        x = self.bn2(x)
        x = F.leaky_relu(x)
        
        # Global Pooling
        x = global_mean_pool(x, batch)
        
        # Final Prediction
        return self.fc(x)

def train_and_evaluate(dataset, test_size=0.2, batch_size=32, epochs=50, learning_rate=0.001):
    # Split dataset
    train_dataset, test_dataset = train_test_split(dataset, test_size=test_size, stratify=dataset.y, random_state=42)
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)
    
    # Device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Model
    model = ImprovedBrainNetworkGNN(num_features=dataset[0].x.shape[1]).to(device)
    
    # Optimizer and Loss
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    criterion = nn.BCEWithLogitsLoss()
    
    # Training Loop
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            
            output = model(batch)
            loss = criterion(output, batch.y.float().view(-1, 1))
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")
    
    # Evaluation
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            output = torch.sigmoid(model(batch)).cpu()
            preds = (output > 0.5).numpy().flatten()
            labels = batch.y.cpu().numpy()
            
            all_preds.extend(preds)
            all_labels.extend(labels)
    
    # Calculate Metrics
    print("\nTest Set Performance:")
    print("Accuracy:", accuracy_score(all_labels, all_preds))
    print("Precision:", precision_score(all_labels, all_preds))
    print("Recall:", recall_score(all_labels, all_preds))
    print("F1 Score:", f1_score(all_labels, all_preds))
    
    # Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    print("\nConfusion Matrix:\n", cm)

def main():
    # Dataset paths
    dataset_path = r'path/to/your/mat_files_root_directory'
    response_path = r'path/to/your/labels.csv'
    
    # Create dataset
    dataset = ImprovedBrainNetworkDataset(
        root=dataset_path,
        response_path=response_path
    )
    
    # Train and Evaluate
    train_and_evaluate(dataset)

if __name__ == "__main__":
    main()